# Movies Metadata — Exploration & Validation

This notebook loads the cleaned movies metadata dataset, validates the cleaning pipeline,
and performs exploratory data analysis (EDA).

### Goals:
- Confirm that the cleaned dataset is correctly formatted
- Explore key variables (genres, runtime, budget, revenue, popularity)
- Generate visualizations for insights
- Export charts to the `reports/` folder


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

clean_path = "../data/cleaned/movies_metadata_clean.parquet"

df = pd.read_parquet(clean_path)
df.head()


,id,title,release_date,genres,budget,revenue,runtime,vote_average,vote_count,popularity,original_language,production_countries,production_companies,cast,director,overview
0,1127164,Beetlejuice Beetlejuice,2024-09-04,"[Comedy, Fantasy, Horror]",100000000,451900435,105,6.999,2681,10.3224,en,[United States of America],"[Plan B Entertainment, Tim Burton Productions,...","[Michael Keaton, Winona Ryder, Catherine O'Har...",Tim Burton,The ghost with the most is back.
1,493529,Dungeons & Dragons: Honor Among Thieves,2023-03-23,"[Adventure, Fantasy, Comedy]",151000000,208200000,134,7.300,4097,9.2202,en,"[Canada, United States of America]","[Entertainment One, Paramount Pictures]","[Chris Pine, Michelle Rodriguez, Justice Smith...",John Francis Daley,A charming thief and a band of unlikely advent...
2,177677,Mission: Impossible - Rogue Nation,2015-07-28,"[Action, Adventure]",150000000,682716636,131,7.200,9228,14.1211,en,[United States of America],"[Paramount Pictures, Bad Robot, Odin, Skydance...","[Tom Cruise, Jeremy Renner, Simon Pegg, Rebecc...",Christopher McQuarrie,Ethan and team take on their most impossible m...
3,1064028,Subservience,2024-08-15,"[Science Fiction, Horror, Thriller]",4400000,262793,105,6.658,815,10.6791,en,[United States of America],"[Millennium Media, Campbell Grobman Films, XYZ...","[Megan Fox, Michele Morrone, Madeline Zima, Ma...",SK Dale,"With his wife out sick, a struggling father br..."
4,1405338,Demon City,2025-02-26,"[Action, Crime, Fantasy, Thriller]",0,0,106,7.034,267,12.8532,ja,[Japan],"[Netflix, AMUSE, Digital Frontier]","[Toma Ikuta, Masahiro Higashide, Miou Tanaka, ...",Seiji Tanaka,Framed for his family's murder and left for de...


## Validate Cleaning Pipeline

We check:
- Data types
- Missing values
- List columns
- Text columns
- Basic statistics


In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
# Check list columns
list_columns = ["genres", "cast", "production_companies", "production_countries"]
for col in list_columns:
    print(col, type(df[col].iloc[0]))

In [ ]:
df.describe(include="all")

### Genre Analysis

#### Count genres

In [ ]:
genre_counts = Counter([g for sublist in df["genres"] for g in sublist])
genre_counts



In [ ]:
plt.figure(figsize=(12,6))
plt.bar(genre_counts.keys(), genre_counts.values(), color="skyblue")
plt.xticks(rotation=45)
plt.title("Genre Frequency")
plt.ylabel("Count")
plt.show()

### Cast Analysis

#### Most frequent actors

In [ ]:
cast_counts = Counter([c for sublist in df["cast"] for c in sublist])
cast_counts.most_common(20)

In [ ]:
top_cast = dict(cast_counts.most_common(20))

plt.figure(figsize=(12,6))
plt.bar(top_cast.keys(), top_cast.values(), color="orange")
plt.xticks(rotation=75)
plt.title("Top 20 Most Frequent Actors")
plt.ylabel("Appearances")
plt.show()

### Runtime Analysis

#### Distribution

In [ ]:
plt.figure(figsize=(10,6))
df["runtime"].hist(bins=40, color="purple")
plt.title("Runtime Distribution")
plt.xlabel("Minutes")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=df["runtime"], color="purple")
plt.title("Runtime Outliers")
plt.show()

### Budget & Revenue Analysis

#### Scatter plot

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df["budget"], df["revenue"], alpha=0.3)
plt.title("Budget vs Revenue")
plt.xlabel("Budget")
plt.ylabel("Revenue")
plt.show()

#### ROI (Return on Investment)

In [ ]:
df["roi"] = (df["revenue"] - df["budget"]) / df["budget"].replace(0, np.nan)
df["roi"].describe()

#### ROI distribution

In [ ]:
df["roi"].hist(bins=40, figsize=(10,6), color="green")
plt.title("ROI Distribution")
plt.show()

### Popularity Analysis

In [ ]:
plt.figure(figsize=(10,6))
df["popularity"].hist(bins=40, color="teal")
plt.title("Popularity Distribution")
plt.xlabel("Popularity Score")
plt.ylabel("Frequency")
plt.show()

### Correlation Analysis

In [ ]:
numeric_cols = ["budget", "revenue", "runtime", "vote_average", "vote_count", "popularity"]

plt.figure(figsize=(10,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

### Outlier Detection

#### Budget outliers

In [25]:
df[df["budget"] > df["budget"].quantile(0.99)][["title", "budget"]].head()

,title,budget
33,The Electric State,320000000


### Revenue outliers

In [26]:
df[df["revenue"] > df["revenue"].quantile(0.99)][["title", "revenue"]].head()

,title,revenue
30,Despicable Me 2,970766005


### Export Charts to `reports/`

In [35]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Ensure reports folder exists
os.makedirs("reports", exist_ok=True)

# ============================
# 1. Runtime Distribution
# ============================
plt.figure(figsize=(10,6))
df["runtime"].hist(bins=40, color="purple", edgecolor="black")
plt.title("Runtime Distribution")
plt.xlabel("Runtime (minutes)")
plt.ylabel("Frequency")
plt.savefig("reports/runtime_distribution.png", dpi=300, bbox_inches="tight")
plt.close()

# ============================
# 2. Genre Frequency
# ============================
genre_counts = Counter([g for sublist in df["genres"] for g in sublist])
plt.figure(figsize=(12,6))
plt.bar(genre_counts.keys(), genre_counts.values(), color="skyblue")
plt.xticks(rotation=45)
plt.title("Genre Frequency")
plt.ylabel("Count")
plt.savefig("reports/genre_frequency.png", dpi=300, bbox_inches="tight")
plt.close()

# ============================
# 3. Budget vs Revenue
# ============================
plt.figure(figsize=(10,6))
plt.scatter(df["budget"], df["revenue"], alpha=0.3)
plt.title("Budget vs Revenue")
plt.xlabel("Budget")
plt.ylabel("Revenue")
plt.savefig("reports/budget_vs_revenue.png", dpi=300, bbox_inches="tight")
plt.close()

# ============================
# 4. Popularity Distribution
# ============================
plt.figure(figsize=(10,6))
df["popularity"].hist(bins=40, color="teal", edgecolor="black")
plt.title("Popularity Distribution")
plt.xlabel("Popularity Score")
plt.ylabel("Frequency")
plt.savefig("reports/popularity_distribution.png", dpi=300, bbox_inches="tight")
plt.close()

# ============================
# 5. Correlation Heatmap
# ============================
numeric_cols = ["budget", "revenue", "runtime", "vote_average", "vote_count", "popularity"]
plt.figure(figsize=(10,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.savefig("reports/correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.close()

# ============================
# 6. Top Actors
# ============================
cast_counts = Counter([c for sublist in df["cast"] for c in sublist])
top_cast = dict(cast_counts.most_common(20))
plt.figure(figsize=(12,6))
plt.bar(top_cast.keys(), top_cast.values(), color="orange")
plt.xticks(rotation=75)
plt.title("Top 20 Most Frequent Actors")
plt.ylabel("Appearances")
plt.savefig("reports/top_actors.png", dpi=300, bbox_inches="tight")
plt.close()

"All charts exported successfully to the reports/ folder."


'All charts exported successfully to the reports/ folder.'